In [2]:
!pip install transformers[torch] datasets[audio] accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.5 MB/s eta 0:00:00


In [3]:
import os

# Paste your token here (it should be the long string starting with KGAT_)
os.environ['KAGGLE_API_TOKEN'] = '<use ur kaggle token for downloading>'

# Now we run the download commands directly
!kaggle datasets download -d uwrfkaggler/ravdess-emotional-speech-audio
!unzip -q ravdess-emotional-speech-audio.zip -d ravdess_data

print("Data downloaded and unzipped successfully!")

Dataset URL: https://www.kaggle.com/datasets/uwrfkaggler/ravdess-emotional-speech-audio
License(s): CC-BY-NC-SA-4.0
100% 429M/429M [00:29<00:00, 15.4MB/s]

Data downloaded and unzipped successfully!


In [4]:
import pandas as pd
import os
from datasets import Dataset, Audio

# This dictionary decodes the RAVDESS filename standard
# The 3rd number in the filename (e.g., 03-01-05) is the emotion.
emotion_map = {
    '01': 'neutral', '02': 'calm', '03': 'happy', '04': 'sad',
    '05': 'angry', '06': 'fearful', '07': 'disgust', '08': 'surprised'
}

data = []
# We walk through all folders to find the .wav files
for root, dirs, files in os.walk("ravdess_data"):
    for file in files:
        if file.endswith(".wav") and not file.startswith("._"):
            path = os.path.join(root, file)
            # Example filename: 03-01-01-01-01-01-01.wav
            parts = file.split("-")
            if len(parts) > 2:
                emotion_id = parts[2]
                data.append({"audio": path, "label": emotion_map[emotion_id]})

# Create the dataset
df = pd.DataFrame(data)
dataset = Dataset.from_pandas(df)

# CRITICAL: Resample to 16kHz for Wav2Vec 2.0
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

# Split into Training (90%) and Testing (10%)
dataset = dataset.train_test_split(test_size=0.1)

print(f"Dataset created! Sample entry: {dataset['train'][0]}")

Dataset created! Sample entry: {'audio': <datasets.features._torchcodec.AudioDecoder object at 0x795c4a4beb10>, 'label': 'sad'}


In [6]:
from transformers import AutoFeatureExtractor

# 1. Load the Wav2Vec 2.0 Feature Extractor
model_id = "facebook/wav2vec2-base"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)

# 2. Define the Preprocessing Function
def preprocess_function(examples):
    # Load the audio arrays
    audio_arrays = [x["array"] for x in examples["audio"]]

    # Process the audio: pad/truncate all clips to 1 second (16000 samples)
    # This ensures every batch has the same shape for the GPU
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=16000,
        truncation=True,
        padding="max_length"
    )
    return inputs

# 3. Process the Dataset (this turns the .wav files into mathematical arrays)
encoded_dataset = dataset.map(preprocess_function, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Map:   0%|          | 0/288 [00:00<?, ? examples/s]

In [8]:
from transformers import AutoModelForAudioClassification, TrainingArguments, Trainer
import numpy as np

# 1. Set up Labels
# RAVDESS has 8 classes. We need to map the numbers back to names.
unique_labels = sorted(list(set(dataset["train"]["label"])))
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for i, label in enumerate(unique_labels)}


# 1. Map the string labels ('happy', 'sad') to their ID numbers (0, 1, 2...)
def label_to_id(examples):
    examples["label"] = [label2id[l] for l in examples["label"]]
    return examples

final_dataset = encoded_dataset.map(label_to_id, batched=True)

# 2. Remove the 'audio' dictionary and keep only the math (input_values)
cols_to_remove = final_dataset["train"].column_names
cols_to_keep = ["input_values", "label"]
final_cols_to_remove = [c for c in cols_to_remove if c not in cols_to_keep]

final_dataset = final_dataset.remove_columns(final_cols_to_remove)

# 3. Tell the dataset to act like PyTorch tensors
final_dataset.set_format("torch")

print("Data is now cleaned and formatted for the GPU!")

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Map:   0%|          | 0/288 [00:00<?, ? examples/s]

Data is now cleaned and formatted for the GPU!


In [9]:


# 2. Load the Model with a classification head
model = AutoModelForAudioClassification.from_pretrained(
    model_id,
    num_labels=len(unique_labels),
    label2id=label2id,
    id2label=id2label
)

# 3. Define Training Arguments
training_args = TrainingArguments(
    output_dir="voice_emotion_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4, # Keep this small so Colab doesn't crash
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=10,
    push_to_hub=False,
)

# 4. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_dataset["train"],  # Use the cleaned version
    eval_dataset=final_dataset["test"],    # Use the cleaned version
    processing_class=feature_extractor,
)

# 5. START TRAINING
trainer.train()

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
projector.weight             | MISSING    | 
projector.bias               | MISSING    | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss
1,2.069764,2.063331
2,2.046040,2.048831
3,2.073187,2.051836
4,2.035299,2.009944
5,1.939932,1.968879


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3240, training_loss=2.052996316956885, metrics={'train_runtime': 504.3879, 'train_samples_per_second': 25.695, 'train_steps_per_second': 6.424, 'total_flos': 1.1766099750912e+17, 'train_loss': 2.052996316956885, 'epoch': 5.0})

In [14]:
import torch
import librosa
import numpy as np

# 1. Pick a sample
sample = dataset["test"][0]
audio_path = sample["audio"]

# 2. Get the array directly from the sample (it's already at 16kHz!)
# In your dataset, sample['audio'] is a dictionary: {'array': [...], 'path': '...', 'sampling_rate': 16000}
speech_array = sample["audio"]["array"]
# 2. Use the pipeline to predict
from transformers import pipeline
# Use 'feature_extractor' explicitly
classifier = pipeline(
    "audio-classification",
    model=model,
    feature_extractor=feature_extractor, # Change this from processing_class
    device=0
)

# Test it
result = classifier(speech_array)
print(f"Result: {result}")

print(f"True Emotion: {sample['label']}")
print(f"Predicted Emotion: {result[0]['label']} (Confidence: {result[0]['score']:.2f})")


Result: [{'score': 0.21747873723506927, 'label': 'fearful'}, {'score': 0.1700243353843689, 'label': 'angry'}, {'score': 0.1455235332250595, 'label': 'happy'}, {'score': 0.13852080702781677, 'label': 'sad'}, {'score': 0.11432277411222458, 'label': 'surprised'}, {'score': 0.10602916032075882, 'label': 'disgust'}, {'score': 0.07784334570169449, 'label': 'calm'}, {'score': 0.030257297679781914, 'label': 'neutral'}]
True Emotion: angry
Predicted Emotion: fearful (Confidence: 0.22)


In [16]:
# This will run the classifier on the first 20 samples and tell you the accuracy
correct = 0
total = 200

for i in range(total):
    s = dataset["test"][i]
    pred = classifier(s["audio"]["array"])[0]["label"]
    if pred == s["label"]:
        correct += 1

print(f"Test Accuracy: {(correct/total)*100}%")

Test Accuracy: 14.499999999999998%


In [17]:
# We use 64,000 because 16,000 samples/sec * 4 seconds = 64,000
MAX_DURATION = 64000

def preprocess_function_long(examples):
    audio_arrays = [x["array"] for x in examples["audio"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=MAX_DURATION,
        truncation=True,
        padding="max_length"
    )
    return inputs

# Re-map the dataset with the longer window
encoded_dataset_long = dataset.map(preprocess_function_long, batched=True)

# Clean it up exactly like before
final_dataset_long = encoded_dataset_long.map(label_to_id, batched=True)
final_dataset_long = final_dataset_long.remove_columns([c for c in final_dataset_long["train"].column_names if c not in ["input_values", "label"]])
final_dataset_long.set_format("torch")

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Map:   0%|          | 0/288 [00:00<?, ? examples/s]

Map:   0%|          | 0/2592 [00:00<?, ? examples/s]

Map:   0%|          | 0/288 [00:00<?, ? examples/s]

In [18]:
training_args_long = TrainingArguments(
    output_dir="voice_emotion_long",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=2, # Reduced to fit 4s audio in memory
    gradient_accumulation_steps=4, # This 'simulates' a batch size of 8
    num_train_epochs=8,            # Increased epochs to give it more time to learn
    weight_decay=0.01,
    logging_steps=10,
    load_best_model_at_end=True,
)

trainer_long = Trainer(
    model=model,
    args=training_args_long,
    train_dataset=final_dataset_long["train"],
    eval_dataset=final_dataset_long["test"],
    processing_class=feature_extractor,
)

trainer_long.train()

Epoch,Training Loss,Validation Loss
1,7.300938,1.791037
2,6.275651,1.607858
3,6.546535,1.566374
4,5.504101,1.357596
5,4.558788,1.141877
6,3.327761,0.917934
7,2.844771,0.763225
8,2.677099,0.696544


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2592, training_loss=5.424856992599405, metrics={'train_runtime': 2357.3098, 'train_samples_per_second': 8.796, 'train_steps_per_second': 1.1, 'total_flos': 7.53030384058368e+17, 'train_loss': 5.424856992599405, 'epoch': 8.0})

In [19]:
# Create a new pipeline with the long-window model
classifier_long = pipeline(
    "audio-classification",
    model=model,
    feature_extractor=feature_extractor,
    device=0
)

correct = 0
total = 100 # Testing 100 samples to get a solid percentage

for i in range(total):
    s = final_dataset_long["test"][i]
    # In final_dataset_long, the math is in 'input_values'
    # We convert to numpy for the pipeline
    input_data = s["input_values"].numpy()
    pred = classifier_long(input_data)[0]["label"]

    # Map the numerical label back to the string name for comparison
    true_label = id2label[s["label"].item()]

    if pred == true_label:
        correct += 1

print(f"New 4-Second Accuracy: {(correct/total)*100}%")

New 4-Second Accuracy: 85.0%


In [20]:
# Save the winning model
trainer_long.save_model("./final_emotion_model_85pct")

# Zip it so you can download it to your PC/Google Drive
import shutil
shutil.make_archive('voice_model_final', 'zip', 'final_emotion_model_85pct')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

'/content/voice_model_final.zip'

In [25]:
import librosa
import torch
from transformers import pipeline

# 1. Path to your downloaded file
file_path = "angry.wav"

# 2. Load and Resample to 16kHz
# This is crucial because Wav2Vec was trained on 16kHz audio
audio_input, _ = librosa.load(file_path, sr=16000)

# 3. Use the classifier you just trained
# (Make sure 'classifier_long' is still in your memory)
result = classifier_long(audio_input)

# 4. Show the results
print(f"--- Analysis for: {file_path} ---")
print(f"Top Prediction: {result[0]['label']} ({result[0]['score']*100:.1f}%)")

print("\nFull Breakdown:")
for r in result:
    print(f"- {r['label']}: {r['score']*100:.1f}%")

--- Analysis for: angry.wav ---
Top Prediction: angry (43.1%)

Full Breakdown:
- angry: 43.1%
- surprised: 41.9%
- fearful: 5.6%
- happy: 4.0%
- neutral: 1.9%
- calm: 1.6%
- disgust: 1.3%
- sad: 0.5%
